In [18]:
import pandas as pd
import numpy as np
 
 
url = "https://raw.githubusercontent.com/mricardo89/data-mining/main/Unit-II/datasets/titanic.csv"
df = pd.read_csv(url)

A - Tasa de sobrevivencia

In [19]:
df['Survived'].value_counts(normalize=True)

Survived
0    0.616162
1    0.383838
Name: proportion, dtype: float64

Pregunta A:

--> Sobrevivió el 38.3838% de los pasajeros del barco. Más de la mitad (61.6162%) no sobrevivió.

B - Sobrevivencia en Mujeres/Hombres

In [20]:
sobrevivientes = df.groupby('Sex')['Survived'].mean()
print(sobrevivientes)


Sex
female    0.742038
male      0.188908
Name: Survived, dtype: float64


Pregunta B: 

--> Sobrevivió el 74.2% de las mujeres vs. un 18.89% de hombres
Se puede decir que el código se respetó.

C - Cuartiles en Tarifas

In [21]:
df['Fare'].quantile([0.25, 0.75])

0.25     7.9104
0.75    31.0000
Name: Fare, dtype: float64

Calculando IQR y rango intercuartilico

In [22]:
q1 = df['Fare'].quantile(0.25)
q3 = df['Fare'].quantile(0.75)
iqr = q3-q1
print("IQR =", iqr)

lim_sup = q3 + 1.5 * iqr
print("Limite superior aceptable =", lim_sup)

IQR = 23.0896
Limite superior aceptable = 65.6344


In [23]:
outliers = df[df['Fare']>lim_sup]
print("Cantidad de pasajeros que pagaron por encima del limite superior aceptable:", len(outliers))

Cantidad de pasajeros que pagaron por encima del limite superior aceptable: 116


In [24]:
print(outliers['Pclass'].value_counts())

Pclass
1    104
3      7
2      5
Name: count, dtype: int64


Pregunta C:
--> La mayoría pertenecía a la primera clase: 104 de 116

In [25]:
promedio = df['Fare'].mean()
print("Promedio: ", promedio)

Promedio:  32.204207968574636


In [26]:
mediana = df['Fare'].median()
print("Mediana: ", mediana)

Mediana:  14.4542


Pregunta D:

1. ¿Qué significa que la media sea tan superior a la mediana? 
--> Que el promedio/media sea mayor que la mediana (el doble aprox.) nos dice que existen outliers a la derecha, y por no tienen muchos registros de esas cantidades la mediana se queda cercana a los datos más recurrentes pero si alcanzan a sesgar el resultado de la media.

2.  ¿Cómo afectará esta asimetría al aprendizaje del modelo KNN?
--> Si estos ouliers toman lugar en un plano, la escala de distancias se verían afectada y los registros con los valores menores donde se concentra la gran mayoría de los datos no tendrian gran protagonismo en este modelo.

In [27]:
df.groupby('Survived')['Age'].mean()

Survived
0    30.626179
1    28.343690
Name: Age, dtype: float64

Pregunta F:
¿Qué sesgo estadístico estamos introduciendo involuntariamente en el resultado de esa función y cómo afectaría la inferencia de nuestro modelo?
--> Es un tipo de sesgo por selección ya que el resultado no representa a toda la población.

Pregunta G:
¿Deberíamos eliminar estas filas (outliers en Fare) con .drop() antes de entrenar nuestro modelo? 
--> Yo opino que debido a que hablamos de un barco de lujo, no deberíamos eliminar los registros de las tarifas más caras ya que estaríamos borrando la existencia de personas que compraron boletos en tarifas de lujo.

H - Mr, Mrs, Miss y Master

In [28]:
df['Name'].head(10)

0                              Braund, Mr. Owen Harris
1    Cumings, Mrs. John Bradley (Florence Briggs Th...
2                               Heikkinen, Miss. Laina
3         Futrelle, Mrs. Jacques Heath (Lily May Peel)
4                             Allen, Mr. William Henry
5                                     Moran, Mr. James
6                              McCarthy, Mr. Timothy J
7                       Palsson, Master. Gosta Leonard
8    Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)
9                  Nasser, Mrs. Nicholas (Adele Achem)
Name: Name, dtype: object

Como podemos observar, los nombres de los passajeros llevan un titulo distintivo que refleja su edad.

Pregunta H: ¿Por qué imputar las edades faltantes basándose en la mediana del "Título" (ej. "Master" = niño, "Mr" = adulto) sería estadísticamente superior y reduciría el error de nuestro futuro modelo, en comparación con usar la mediana global?

--> Porque estaríamos rellenando huecos basándonos en datos reales o aproximados a la realidad, sin la necesidad de predecir y suponer las edades faltantes con la mediana.

I - Varianza en 'Survived'

In [34]:
varianza = np.var(df['Survived'], ddof=1)
print(varianza)

0.2367722165474984


Pregunta I:
1. ¿Qué significaría si la varianza de esta variable fuera exactamente 0.0? 
--> Si la varianza diera como resultado 0, significa que todos los datos son iguales y por ende el promedio sería el mismo que todos los datos, dando una nula dispersion respecto al promedio.

2. ¿Qué pasaría si intentan entrenar un algoritmo de clasificación con un dataset donde la variable de respuesta tiene varianza 0.0?
--> Aprendería que siempre se tendrá el mismo resultado, prediciéndolo en cada ocasion y no sabría como adaptarse a la introduccion de nuevos datos.

J - Agrupación

In [37]:
df.groupby(['Pclass', 'Sex', 'Embarked'])['PassengerId'].count()

Pclass  Sex     Embarked
1       female  C            43
                Q             1
                S            48
        male    C            42
                Q             1
                S            79
2       female  C             7
                Q             2
                S            67
        male    C            10
                Q             1
                S            97
3       female  C            23
                Q            33
                S            88
        male    C            43
                Q            39
                S           265
Name: PassengerId, dtype: int64

Pregunta J:
¿Qué fenómeno perjudicial (sobreajuste o subajuste) ocurrirá inevitablemente si dejamos que el modelo aprenda reglas basadas en esos grupos de 1 solo pasajero?
--> El fenomeno que ocurre es el sobreajuste ya que al no tener mas datos de los cuales aprender, se aprendera unicamente esa respuesta y pensara que es 100% correcta. 